# ColabDFT ESP — 静電ポテンシャルの表示

入力した構造のまま、[GPU4PySCF](https://github.com/pyscf/gpu4pyscf) で DFT の一点計算を行い、**静電ポテンシャル（ESP）を等密度面に色付けして 3D 表示** するノートブックです。構造の最適化はしません。コードを書く必要はありません。UMA は使わないので、Hugging Face のトークンはいりません。

## 使い方

1. メニューの **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ
2. 下の **1. Input form** で出力名、電荷、スピン多重度、汎関数、基底関数、等密度面の値を指定する
3. **ランタイム → すべて実行** を押す
4. **2. File selection and checks** にファイル選択ボタンが出るので、xyz または Gaussian の gjf / com を選ぶ。入力を「draw」にしたときは構造エディタが出るので、描いて「Use this structure」を押す。あとは放置できます
5. 下のセルで ESP の 3D 表示とエネルギーを確認し、最後のセルで結果のZIPをダウンロードする

gjfからは構造だけを読み込みます。計算方法はフォームの値を使い、電荷と多重度もフォームの値を使います。xyz に複数の構造が入っている場合は先頭だけを計算します。

2回目以降は手順2〜5を繰り返すだけです（インストールは自動でスキップされます）。ファイル選択で Cancel upload を押すと、前回のファイルをそのまま使います。「draw」では前回描いた構造が最初に表示されます。

## 出力（ZIPの中身）

| ファイル | 内容 |
|---|---|
| `<出力名>_spe.xyz` | 計算した構造。コメント行にエネルギーと計算条件 |
| `<出力名>_esp.cube` | 格子上の ESP（a.u.）。GaussView などで開ける |
| `<出力名>_density.cube` | 格子上の電子密度（e/Bohr³）。GaussView では、これの等値面に `_esp.cube` を色付けする |
| `<出力名>_dft.csv` | DFT 一点計算のエネルギー |
| `<出力名>_log.txt` | 入力条件、ソフトのバージョン、計算結果、面の上の ESP の最小・最大 |

## 注意事項

- 色は **赤が負**（+1 の電荷が引き寄せられる、電子が豊富な側）、**青が正** です。ESP の数値は +1 の点電荷あたりの kcal/mol です。
- 色の範囲を「自動」にすると分子ごとに範囲が変わります。複数の分子の色を比べるときは、同じ数値を指定してください。
- 3D 表示はノートが重くならないよう格子を間引いています。ZIP の cube は間引いていません。
- 表示される最小・最大は、面の近くの格子点の値です。格子の細かさで少し変わります。
- ECP を使う元素（Rb 以降）を含むときの密度は、価電子の密度です。
- DFT は GPU4PySCF（密度フィッティング）で計算します。T4 は倍精度の計算が遅いため、原子数が多いときや def2-TZVPD では時間がかかります。GPU が使えないときは CPU の PySCF で計算します。
- 閉殻（多重度1）は RKS、開殻は UKS で計算します。
- 「draw」では、描いた構造に水素を付け、RDKit（ETKDG で 10 個の配座を作り、MMFF94 で最適化して一番低いもの）で 3D の構造にします。構造の最適化はしないので、正確な構造で見たいときは ColabUMA Opt で最適化した xyz を使ってください。電荷はフォームの値を使います（描いた形式電荷と違えば注意を出します）。
- 扱える元素は H〜Rn（原子番号1〜86）です。周期系（結晶・表面）には対応していません。
- UMA で構造を最適化してから一点計算するときは [ColabUMA_Opt](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_Opt.ipynb) を使ってください。
- エネルギーだけでよいときは [ColabDFT_SPE](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabDFT_SPE.ipynb) のほうが速く終わります。

## ライセンス

- このノートと共通コード（`colabuma`）：[MIT License](https://github.com/RChibaChiba/ColabChem/blob/main/LICENSE)
- 実行時にインストールするライブラリは、それぞれのライセンスに従います。一覧は [README のライセンス](https://github.com/RChibaChiba/ColabChem/blob/main/README.ja.md#ライセンス) にあります

## 引用

論文で使う場合は、以下を引用してください。

- PySCF（Q. Sun *et al.*, *J. Chem. Phys.* **153**, 024109 (2020)）
- GPU4PySCF（X. Wu *et al.*, arXiv:2404.09452 (2024)）

In [ ]:
#@title 1. Input form { display-mode: "form" }
#@markdown When the form is filled in, choose **Runtime → Run all**. Pick an xyz or gjf file in the next cell; after that you can leave it running.

#@markdown ### Output
output_name = "result"  #@param {type:"string"}

#@markdown ### Structure
#@markdown file: choose an xyz or gjf file. draw: draw the structure in the editor shown in the next cell (hydrogens are added and RDKit makes it 3D).
input_mode = "file"  #@param ["file", "draw"]
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### DFT (gpu4pyscf)
#@markdown ωB97M-V/def2-TZVPD is the level of theory of UMA's training data (OMol25). def2-TZVPD can take tens of minutes per point on a T4.
dft_functional = "B3LYP-D3(BJ)"  #@param ["ωB97M-V (same as OMol25)", "B3LYP-D3(BJ)", "PBE0-D3(BJ)", "M06-2X", "r2SCAN-D4"]
dft_basis = "def2-SVP"  #@param ["def2-SVP", "def2-TZVP", "def2-TZVPD"]

#@markdown ### ESP view
#@markdown Density of the isosurface (e/Bohr³). 0.001 is often used as the size of the molecule; 0.002 gives a surface slightly further in.
isosurface = 0.002  #@param {type:"number"}
#@markdown Grid quality. A finer grid looks better but takes longer and makes larger files.
grid_quality = "standard"  #@param ["coarse", "standard", "fine"]
#@markdown Color range ± (kcal/mol). 0 means automatic. Use the same value when comparing molecules.
esp_color_range = 0  #@param {type:"number"}

In [ ]:
#@title 2. File selection and checks { display-mode: "form" }
#@markdown File selection (or drawing the structure) happens here. You can leave the install and calculations after this running.

import os
import subprocess
import sys

COLABUMA_REF = "main"  # GitHub branch or tag of the shared code (colabuma package)


def _use_colabuma(ref):
    # Use colabuma next to the notebook if it is there, otherwise fetch it from GitHub
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabChem" if os.path.isdir("/content") else os.path.expanduser("~/ColabChem")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabChem.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"Could not fetch the ColabChem shared code from GitHub (ref {ref}).\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import dft, env, esp, output, structure, ui
from colabuma.ui import ok

ui.install_error_handler()

structure.check_multiplicity(multiplicity)
functional, basis = dft.parse_method(dft_functional, dft_basis)
esp.check_form(isosurface, esp_color_range, grid_quality)
job = dft.Job(structure.out_prefix(output_name), charge, multiplicity, functional, basis)
env.check_gpu()
upload_name, upload_data = structure.get_input(input_mode)
ok("Input checked. You can leave it running from here.")

In [ ]:
#@title 3. Setup (install) { display-mode: "form" }
DEVICE = env.setup_dft(drawing=structure.is_drawing(input_mode))

In [ ]:
#@title 4. Prepare the structure { display-mode: "form" }
atoms, source_desc = structure.read_structure(upload_name, upload_data, charge, multiplicity, uma=False)
structure.show_numbered(atoms)
ok(f"Structure ready: {atoms.get_chemical_formula()} ({len(atoms)} atoms).")

In [ ]:
#@title 5. DFT { display-mode: "form" }
dft_res = dft.run([("Input structure", atoms)], charge, multiplicity, functional, basis, keep_density=True)
files, dft_log = dft.report(dft_res, None, geometry="input structure")
_p = dft_res["points"][0]
files["spe.xyz"] = structure.xyz_block(atoms.get_chemical_symbols(), atoms.positions,
                                       f"energy_Ha={_p['energy']:.10f} converged={_p['converged']} "
                                       f"method={functional}/{basis} charge={charge} multiplicity={multiplicity}")

In [ ]:
#@title 6. Compute the ESP { display-mode: "form" }
esp_res = esp.compute(dft_res["points"][0], atoms, isosurface, grid_quality)

In [ ]:
#@title 7. Show the ESP { display-mode: "form" }
#@markdown Red is negative and blue is positive. Rotate and zoom with the mouse. Atom numbers are shown in "4. Prepare the structure".
_f, esp_log = esp.report(esp_res, esp_color_range)
files.update(_f)

In [ ]:
#@title 8. Save { display-mode: "form" }
files["log.txt"] = output.log_text("ColabDFT ESP log", source_desc, job, [dft_log, esp_log])
zip_path = output.save_zip(job.out_prefix, files)